# 02 — Preprocessing & Feature Engineering

# Predicting Gallstone Disease from Clinical + Bioimpedance

**Domain:** Healthcare | **Task:** Binary Classification | **Data:** Tabular  
**Dataset:** `data/dataset-uci.csv`  
**Target column:** `Gallstone Status` (Yes[1]/No[0])

Run the cells top-to-bottom. These notebooks avoid seaborn per instructions and use pure matplotlib for charts.

## Objectives
- Clean column names (optional)
- Encode target to 0/1
- Optional outlier clipping (robust scaling later handles much of this)
- Train/test split and scaling
- Persist processed arrays for reuse

In [1]:
# Core setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Settings
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

DATA_PATH = "../data/dataset-uci.csv"
RANDOM_STATE = 42

# Load
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(319, 39)


,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,Body Mass Index (BMI),Total Body Water (TBW),Extracellular Water (ECW),Intracellular Water (ICW),Extracellular Fluid/Total Body Water (ECF/TBW),Total Body Fat Ratio (TBFR) (%),Lean Mass (LM) (%),Body Protein Content (Protein) (%),Visceral Fat Rating (VFR),Bone Mass (BM),Muscle Mass (MM),Obesity (%),Total Fat Content (TFC),Visceral Fat Area (VFA),Visceral Muscle Area (VMA) (Kg),Hepatic Fat Accumulation (HFA),Glucose,Total Cholesterol (TC),Low Density Lipoprotein (LDL),High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
0,0,50,0,0,0,0,0,0,185,92.8,27.1,52.9,21.2,31.7,40.0,19.2,80.84,18.88,9,3.7,71.4,23.4,17.8,10.6,39.7,0,102.0,250.0,175.0,40.0,134.0,20.0,22.0,87.0,0.82,112.47,0.0,16.0,33.0
1,0,47,0,1,0,0,0,0,176,94.5,30.5,43.1,19.5,23.6,45.0,32.8,67.20,16.68,15,3.2,60.3,38.8,31.0,18.4,32.7,0,94.0,172.0,108.0,43.0,103.0,14.0,13.0,46.0,0.87,107.10,0.0,14.4,25.0
2,0,61,0,0,0,0,0,0,171,91.1,31.2,47.2,20.1,27.1,43.0,27.3,72.67,16.35,15,3.3,62.9,41.7,24.9,16.2,34.0,0,103.0,179.0,124.0,43.0,69.0,18.0,14.0,66.0,1.25,65.51,0.0,16.2,30.2
3,0,41,0,0,0,0,0,0,168,67.7,24.0,41.4,17.0,24.4,41.0,15.8,84.19,16.90,6,2.9,54.1,9.0,10.7,6.5,29.2,1,69.0,173.0,73.0,59.0,53.0,20.0,12.0,34.0,1.02,94.10,0.0,15.4,35.4
4,0,42,0,0,0,0,0,0,178,89.6,28.3,51.4,20.0,31.4,39.0,20.0,80.02,16.81,8,3.5,68.2,28.6,17.9,10.4,37.4,2,109.0,205.0,154.0,30.0,326.0,27.0,54.0,71.0,0.82,112.47,0.0,16.8,40.6


## Clean column names (optional, reproducible)

In [2]:
def clean_cols(cols):
    return [c.strip().replace(' ', '_').replace('(', '').replace(')', '').replace('%','pct').replace('/','_') for c in cols]

df.columns = clean_cols(df.columns)
assert "Gallstone_Status" in df.columns, "After cleaning, expected 'Gallstone_Status'."
df.head()

,Gallstone_Status,Age,Gender,Comorbidity,Coronary_Artery_Disease_CAD,Hypothyroidism,Hyperlipidemia,Diabetes_Mellitus_DM,Height,Weight,Body_Mass_Index_BMI,Total_Body_Water_TBW,Extracellular_Water_ECW,Intracellular_Water_ICW,Extracellular_Fluid_Total_Body_Water_ECF_TBW,Total_Body_Fat_Ratio_TBFR_pct,Lean_Mass_LM_pct,Body_Protein_Content_Protein_pct,Visceral_Fat_Rating_VFR,Bone_Mass_BM,Muscle_Mass_MM,Obesity_pct,Total_Fat_Content_TFC,Visceral_Fat_Area_VFA,Visceral_Muscle_Area_VMA_Kg,Hepatic_Fat_Accumulation_HFA,Glucose,Total_Cholesterol_TC,Low_Density_Lipoprotein_LDL,High_Density_Lipoprotein_HDL,Triglyceride,Aspartat_Aminotransferaz_AST,Alanin_Aminotransferaz_ALT,Alkaline_Phosphatase_ALP,Creatinine,Glomerular_Filtration_Rate_GFR,C-Reactive_Protein_CRP,Hemoglobin_HGB,Vitamin_D
0,0,50,0,0,0,0,0,0,185,92.8,27.1,52.9,21.2,31.7,40.0,19.2,80.84,18.88,9,3.7,71.4,23.4,17.8,10.6,39.7,0,102.0,250.0,175.0,40.0,134.0,20.0,22.0,87.0,0.82,112.47,0.0,16.0,33.0
1,0,47,0,1,0,0,0,0,176,94.5,30.5,43.1,19.5,23.6,45.0,32.8,67.20,16.68,15,3.2,60.3,38.8,31.0,18.4,32.7,0,94.0,172.0,108.0,43.0,103.0,14.0,13.0,46.0,0.87,107.10,0.0,14.4,25.0
2,0,61,0,0,0,0,0,0,171,91.1,31.2,47.2,20.1,27.1,43.0,27.3,72.67,16.35,15,3.3,62.9,41.7,24.9,16.2,34.0,0,103.0,179.0,124.0,43.0,69.0,18.0,14.0,66.0,1.25,65.51,0.0,16.2,30.2
3,0,41,0,0,0,0,0,0,168,67.7,24.0,41.4,17.0,24.4,41.0,15.8,84.19,16.90,6,2.9,54.1,9.0,10.7,6.5,29.2,1,69.0,173.0,73.0,59.0,53.0,20.0,12.0,34.0,1.02,94.10,0.0,15.4,35.4
4,0,42,0,0,0,0,0,0,178,89.6,28.3,51.4,20.0,31.4,39.0,20.0,80.02,16.81,8,3.5,68.2,28.6,17.9,10.4,37.4,2,109.0,205.0,154.0,30.0,326.0,27.0,54.0,71.0,0.82,112.47,0.0,16.8,40.6


## Encode target and split features/labels

In [3]:
y = df["Gallstone_Status"].astype(int)
X = df.drop(columns=["Gallstone_Status"])
feature_names = X.columns.tolist()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Target distribution:", y.value_counts().to_dict())

X shape: (319, 38) | y shape: (319,)
Target distribution: {0: 161, 1: 158}


## Train/validation/test split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)

Train: (223, 38) Valid: (48, 38) Test: (48, 38)


## Scaling (StandardScaler) — save fitted scaler

In [5]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled  = scaler.transform(X_test)

# Save scaler in models
joblib.dump(scaler, "../data/models/scaler.joblib")

# Save processed arrays in processed folder
np.save("../data/processed/X_train_scaled.npy", X_train_scaled)
np.save("../data/processed/X_valid_scaled.npy", X_valid_scaled)
np.save("../data/processed/X_test_scaled.npy",  X_test_scaled)
np.save("../data/processed/y_train.npy", y_train.values)
np.save("../data/processed/y_valid.npy", y_valid.values)
np.save("../data/processed/y_test.npy",  y_test.values)

# Save feature names in processed folder
pd.Series(feature_names).to_csv("../data/processed/feature_names.csv", index=False)

print("✓ Saved scaled arrays and scaler.")
print(f"  - Scaler: data/models/scaler.joblib")
print(f"  - Processed data: data/processed/")

✓ Saved scaled arrays and scaler.
  - Scaler: data/models/scaler.joblib
  - Processed data: data/processed/


## (Optional) Simple feature creation examples

In [6]:
# Example: BMI is likely already present; if not, show pattern
# if set(['Weight','Height']).issubset(set(X.columns)):
#     X['BMI_calc'] = X['Weight'] / (X['Height']/100)**2

"Add domain features here if needed."

'Add domain features here if needed.'